# VBZ Data Preparation — Polars

Schrittweise Zusammenführung aller Datenschichten zum finalen `vbz_master.parquet`.

**Bibliothek:** Polars (primär) — Pandas nur für kleine Hilfstabellen (GTFS, Meteo, Events)

**Warum Polars?** 4× schneller, 4× weniger RAM — getestet auf den echten 88 Mio Zeilen IST-Daten.

---

In [ ]:
from IPython.display import SVG, display
from pathlib import Path

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'reports').exists():
        break
display(SVG(filename=str(_p / 'reports' / 'vbz_preparation.svg')))

---

## 0 — Setup

In [ ]:
import polars as pl
import pandas as pd
import time, psutil
from pathlib import Path

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        ROOT = _p; break

IST_DIR    = ROOT / 'data' / 'interim' / 'vbz' / 'ist-daten'
GTFS_DIR   = ROOT / 'data' / 'interim' / 'vbz' / 'gtfs'
METEO_DIR  = ROOT / 'data' / 'interim' / 'vbz' / 'meteo'
EVENTS_DIR = ROOT / 'data' / 'interim' / 'vbz'
OUT_DIR    = ROOT / 'data' / 'interim' / 'vbz'

print(f'Root:     {ROOT}')
print(f'IST:      {len(list(IST_DIR.glob("*.parquet"))):,} Parquets')
print(f'RAM frei: {psutil.virtual_memory().available / 1e9:.1f} GB')

---

## Schritt 1 — IST-Daten laden + Rename + Cast

**Polars Lernmoment: `scan_parquet` + `collect()`**
Polars liest alle Parquets lazy ein — erst beim `collect()` werden sie tatsächlich in den RAM geladen.
Intern parallelisiert Polars über alle CPU-Kerne.

**Polars Lernmoment: `rename()` + `with_columns()`**
```python
df.rename({'ALT': 'neu'})                        # Spalten umbenennen
df.with_columns([pl.col('x').cast(pl.Int32)])    # Spalten transformieren (in-place)
pl.col('canceled').eq('true')                    # String 'true'/'false' → bool
```

In [ ]:
t0 = time.time()

df = pl.scan_parquet(str(IST_DIR / '*.parquet')).collect()

# ── Rename: alte Namen → finale Spaltennamen ────────────────────
df = df.rename({
    'BETRIEBSTAG'   : 'operating_date',
    'LINIEN_TEXT'   : 'line_name',
    'BPUIC'         : 'bpuic',
    'ANKUNFTSZEIT'  : 'sta',
    'AN_DELAY'      : 'arrival_delay',
    'ABFAHRTSZEIT'  : 'std',
    'AB_DELAY'      : 'departure_delay',
    'FAELLT_AUS_TF' : 'canceled',
})

# ── Cast: optimierte Datentypen ─────────────────────────────────
df = df.with_columns([
    pl.col('operating_date').str.strptime(pl.Date, '%d.%m.%Y'),
    pl.col('bpuic').cast(pl.Int32),
    pl.col('arrival_delay').cast(pl.Float32),
    pl.col('departure_delay').cast(pl.Float32),
    pl.col('canceled').eq('true'),
    pl.col('line_name').cast(pl.Categorical),
])

print(f'Geladen in {time.time()-t0:.1f}s  |  {len(df):,} Zeilen  |  RAM frei: {psutil.virtual_memory().available / 1e9:.1f} GB')
print(f'Schema: {dict(df.schema)}')
df.head(3)

---

## Schritt 2 — GTFS Stops Lookup laden

Kleine Hilfstabelle → Pandas laden, dann zu Polars konvertieren.

**Warum `gtfs_stops_lookup` und nicht `gtfs_tram_stops`?**
Die `gtfs_tram_stops.parquet` verwendet SLOID-Format als `stop_id` (`ch:1:sloid:90805::0`) —
kein direkter Join mit `bpuic` möglich (0 Matches).
Die Lookup-Tabelle wurde in `vbz-gtfs-data.ipynb` speziell für diesen Join gebaut:
BPUIC aus `stop_url` extrahiert, 1 Zeile pro Haltestelle, mittlere Koordinaten.

**Polars Lernmoment: `pl.from_pandas()`**
```python
pl.from_pandas(df_pandas)   # Pandas → Polars
df_polars.to_pandas()       # Polars → Pandas
```

In [ ]:
stops_pd = pd.read_parquet(GTFS_DIR / 'gtfs_stops_lookup.parquet')

stops_pl = (
    pl.from_pandas(stops_pd[['bpuic', 'stop_name', 'stop_lat', 'stop_lon']])
    .with_columns([
        pl.col('bpuic').cast(pl.Int32),
        pl.col('stop_name').cast(pl.Categorical),
        pl.col('stop_lat').cast(pl.Float32),
        pl.col('stop_lon').cast(pl.Float32),
    ])
)

print(f'{len(stops_pl):,} Haltestellen  |  Schema: {dict(stops_pl.schema)}')
stops_pl.head(3)

---

## Schritt 3 — Join: IST + GTFS

**Polars Lernmoment: `.join()`**
```python
# Gleicher Spaltenname auf beiden Seiten:
df.join(other, on='bpuic', how='left')

# Unterschiedliche Spaltennamen:
df.join(other, left_on='A', right_on='B', how='left')
```
Entspricht SQL: `SELECT * FROM df LEFT JOIN other ON df.bpuic = other.bpuic`

In [ ]:
t0 = time.time()
df = df.join(stops_pl, on='bpuic', how='left')

unmatched = df['stop_name'].null_count()
print(f'Join in {time.time()-t0:.1f}s  |  nicht gematchte bpuic: {unmatched:,} ({unmatched/len(df)*100:.2f}%)')
# Erwartung: 0 unmatched ✓

---

## Schritt 4 — Meteo Master laden

Wir nehmen nur die Spalten die im finalen Master erscheinen.
Drei Spalten werden **gedroppt** (nicht benötigt für MVP): `air_pressure`, `wind_direction`, `wind_speed_vector`.

**Polars Lernmoment: `.select()` mit Ausdrücken**
```python
# select wählt UND transformiert in einem Schritt:
df.select([
    'col_a',
    pl.col('col_b', 'col_c').cast(pl.Float32),
    pl.col('col_d').cast(pl.Int16),
])
```

In [ ]:
meteo_pl = (
    pl.from_pandas(pd.read_parquet(METEO_DIR / 'meteo-master.parquet'))
    .rename({'precipitation_mm': 'precipitation'})
    .select([
        'date_time',
        pl.col('temperature', 'humidity', 'rain_duration',
               'precipitation', 'wind_speed', 'global_radiation').cast(pl.Float32),
        pl.col('flood_intensity').cast(pl.Int16),
    ])
)

print(f'Meteo: {len(meteo_pl):,} Einträge  |  {meteo_pl["date_time"].min()} bis {meteo_pl["date_time"].max()}')
meteo_pl.head(3)

---

## Schritt 5 — Join: IST + Meteo

Temporärer Schlüssel `_h`: `sta` auf volle Stunde abrunden, joinen, dann droppen.

**Polars Lernmoment: Method Chaining**
```python
# Mehrere Operationen in einem Ausdruck:
df = (
    df
    .with_columns(pl.col('sta').dt.truncate('1h').alias('_h'))
    .join(...)
    .drop('_h')
)
```

In [ ]:
t0 = time.time()
df = (
    df
    .with_columns(pl.col('sta').dt.truncate('1h').alias('_h'))
    .join(meteo_pl.rename({'date_time': '_h'}), on='_h', how='left')
    .drop('_h')
)

unmatched = df['temperature'].null_count()
print(f'Join in {time.time()-t0:.1f}s  |  ohne Wetter-Match: {unmatched:,} ({unmatched/len(df)*100:.2f}%)')

---

## Schritt 6 — Events Master laden

Rename direkt beim Laden — der Master bekommt sofort die finalen Spaltennamen.
Events verwenden `operating_date` als Join-Schlüssel — **kein temporärer Schlüssel nötig**,
weil beide Seiten bereits `pl.Date` haben.

In [ ]:
events_pd = pd.read_csv(EVENTS_DIR / 'events-master.csv', sep=';')
events_pd['Datum'] = pd.to_datetime(events_pd['Datum'])

events_pl = (
    pl.from_pandas(events_pd)
    .with_columns(pl.col('Datum').cast(pl.Date))
    .rename({
        'Datum'      : 'operating_date',
        'Event_Name' : 'event_name',
        'Typ'        : 'event_type',
        'Gewichtung' : 'event_size',
        'Ort'        : 'event_location',
    })
    .with_columns([
        pl.col('event_name', 'event_type', 'event_location').cast(pl.Categorical),
        pl.col('event_size').cast(pl.Int8),
    ])
)

print(f'Events: {len(events_pl):,}  |  Typen: {sorted(events_pl["event_type"].unique().to_list())}')
events_pl.head(3)

---

## Schritt 7 — Join: IST + Events

Direkter Join auf `operating_date` — beide Seiten sind `pl.Date`, kein Zwischenschritt.

> **Multi-Event-Tage:** Fällt z.B. Street Parade + FCZ-Spiel auf denselben Tag,
> entstehen nach dem left join **mehrere Zeilen pro Fahrt**.
> Für die EDA: `max(event_size)` pro Tag aggregieren.

In [ ]:
t0 = time.time()
df = df.join(events_pl, on='operating_date', how='left')

event_rows = df['event_name'].is_not_null().sum()
print(f'Join in {time.time()-t0:.1f}s  |  mit Event: {event_rows:,} ({event_rows/len(df)*100:.1f}%)')

---

## Schritt 8 — Qualitätsprüfung

In [ ]:
print(f'Zeilen:  {len(df):,}')
print(f'Spalten: {df.width}')
print(f'RAM:     {df.estimated_size("gb"):.2f} GB')
print()
print('Schema:')
for col, dtype in df.schema.items():
    print(f'  {col:<25} {dtype}')

In [ ]:
print('Null-Quoten (nur Spalten > 0%):')
(
    df.select(pl.all().null_count())
    .transpose(include_header=True, column_names=['null_count'])
    .with_columns((pl.col('null_count') / len(df) * 100).round(2).alias('pct'))
    .filter(pl.col('pct') > 0)
    .sort('pct', descending=True)
)

In [ ]:
# Stichprobe: Event-Tag
df.filter(pl.col('event_name').is_not_null()).head(3)

---

## Schritt 9 — Export

**Polars Lernmoment: `write_parquet()`**
```python
df.write_parquet(path)                               # Polars — Snappy-Kompression
df.to_parquet(path, engine='pyarrow', index=False)   # Pandas-Äquivalent
```

In [ ]:
OUT_PATH = OUT_DIR / 'vbz_master.parquet'
t0 = time.time()
df.write_parquet(str(OUT_PATH))

print(f'Exportiert in {time.time()-t0:.1f}s')
print(f'Dateigröße:   {OUT_PATH.stat().st_size / 1e6:.0f} MB')
print(f'Pfad:         {OUT_PATH}')
print('\n✓ vbz_master.parquet ist bereit.')